In [1]:
!pip install transformers datasets sentencepiece evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [2]:
import json
from tqdm import tqdm
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

In [3]:
from google.colab import files
uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [4]:
def extract_answer(item):
    ans = item.get("answer", {})

    # case 1: structured entity answer
    if isinstance(ans, dict) and "answer" in ans and ans["answer"]:
        obj = ans["answer"][0]

        # entity with labels
        if isinstance(obj, dict):
            if "label" in obj and isinstance(obj["label"], dict):
                if obj["label"].get("en"):
                    return str(obj["label"]["en"])

            if "name" in obj:
                return str(obj["name"])

        # numeric / boolean / string
        return str(obj)

    # fallback mention
    if isinstance(ans, dict) and "mention" in ans:
        return str(ans["mention"])

    return ""


def load_mintaka(path):
    with open(path) as f:
        data = json.load(f)

    questions = []
    answers = []

    for item in data:
        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers, data


train_q, train_a, train_data = load_mintaka("mintaka_train.json")
dev_q, dev_a, dev_data = load_mintaka("mintaka_dev.json")
test_q, test_a, test_data = load_mintaka("mintaka_test.json")

In [5]:
print(train_q[0])
print(train_a[0])

What is the seventh tallest mountain in North America?
Mount Lucania


In [21]:
import requests
import time

def wikidata_search(label):
    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "format": "json"
    }

    try:
        r = requests.get(url, params=params).json()
        if r["search"]:
            return r["search"][0]["id"]
    except:
        pass

    return None


def get_triples(eid, limit=5):
    sparql = f"""
    SELECT ?pLabel ?oLabel WHERE {{
      wd:{eid} ?p ?o .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }} LIMIT {limit}
    """
    url = "https://query.wikidata.org/sparql"
    try:
        r = requests.get(url, params={"query": sparql, "format": "json"}, timeout=20).json()
        triples = []
        for b in r["results"]["bindings"]:
            p = b["pLabel"]["value"]
            o = b["oLabel"]["value"]
            triples.append(f"{eid} {p} {o}")
        return triples
    except:
        return []

In [20]:
import requests
import re

def extract_entity(question):
    # simple heuristic: longest capitalized phrase
    words = question.split()
    candidates = []

    current = []
    for w in words:
        if w[0].isupper():
            current.append(w)
        else:
            if current:
                candidates.append(" ".join(current))
                current = []
    if current:
        candidates.append(" ".join(current))

    if candidates:
        return max(candidates, key=len)
    return None

In [22]:
def build_contexts_triples(data_items, limit=5):
    contexts = []
    entity_cache = {}

    for item in data_items:

        question = item["question"]
        triples_all = []

        # automatic entity detection
        entity_label = extract_entity(question)

        if entity_label:

            if entity_label in entity_cache:
                triples = entity_cache[entity_label]
            else:
                eid = wikidata_search(entity_label)
                triples = get_triples(eid, limit) if eid else []
                entity_cache[entity_label] = triples

            triples_all.extend(triples)

        contexts.append(" ".join(triples_all))

    return contexts

In [23]:
train_ctx = build_contexts_triples(train_data, limit=5)
dev_ctx   = build_contexts_triples(dev_data, limit=5)
test_ctx  = build_contexts_triples(test_data, limit=5)

In [24]:
def create_tuples(questions, contexts, answers):
    inputs = []
    labels = []

    for q, ctx, ans in tqdm(zip(questions, contexts, answers), total=len(questions)):
        inp = f"question: {q} context: {ctx}"
        inputs.append(inp)
        labels.append(ans)   # ALWAYS use gold answer

    return inputs, labels

In [25]:
train_inp, train_lab = create_tuples(train_q, train_ctx, train_a)
dev_inp, dev_lab     = create_tuples(dev_q, dev_ctx, dev_a)
test_inp, test_lab   = create_tuples(test_q, test_ctx, test_a)

100%|██████████| 4000/4000 [00:00<00:00, 1306636.76it/s]


In [26]:
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-base")

In [27]:
from datasets import Dataset

def make_dataset(inputs, labels):
    ds = Dataset.from_dict({
        "input_text": inputs,
        "target_text": labels
    })

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

        labels_tok = tokenizer(
            batch["target_text"],
            truncation=True,
            padding="max_length",
            max_length=32
        )

        model_inputs["labels"] = labels_tok["input_ids"]
        return model_inputs

    ds = ds.map(tokenize, batched=True)

    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return ds

In [28]:
train_dataset = make_dataset(train_inp, train_lab)
dev_dataset   = make_dataset(dev_inp, dev_lab)
test_dataset  = make_dataset(test_inp, test_lab)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

training_args = TrainingArguments(
    output_dir="./t5_kgqa",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("final_t5_kgqa")
tokenizer.save_pretrained("final_t5_kgqa")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_t5_kgqa/tokenizer_config.json', 'final_t5_kgqa/tokenizer.json')

In [ ]:
!zip -r final_t5_kgqa.zip final_t5_kgqa
from google.colab import files
files.download("final_t5_kgqa.zip")

  adding: final_t5_kgqa/ (stored 0%)
  adding: final_t5_kgqa/generation_config.json (deflated 29%)
  adding: final_t5_kgqa/model.safetensors (deflated 10%)
  adding: final_t5_kgqa/config.json (deflated 63%)
  adding: final_t5_kgqa/tokenizer_config.json (deflated 83%)
  adding: final_t5_kgqa/tokenizer.json (deflated 79%)
  adding: final_t5_kgqa/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
def predict_answer(question, context):
    inp = f"question: {question} context: {context}"

    ids = tokenizer(inp, return_tensors="pt").input_ids.to(model.device)

    out = model.generate(ids, max_length=32)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    return text

In [18]:
preds = []
for q, ctx in tqdm(zip(test_q, test_ctx), total=len(test_q)):
    preds.append(predict_answer(q, ctx))

100%|██████████| 4000/4000 [06:34<00:00, 10.15it/s]


In [19]:
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = " ".join(text.split())
    return text


def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


# Hit@1
hit1 = sum(
    normalize(p) == normalize(g)
    for p, g in zip(preds, test_a)
) / len(test_a)

# Hit@5 (same for single prediction)
hit5 = hit1

# MRR (same for single prediction)
mrr = hit1

# F1
f1 = sum(
    f1_score(p, g)
    for p, g in zip(preds, test_a)
) / len(test_a)

# Accuracy
accuracy = hit1

print("Hit@1:", hit1)
print("Hit@5:", hit5)
print("MRR:", mrr)
print("F1:", f1)
print("Accuracy:", accuracy)

Hit@1: 0.17975
Hit@5: 0.17975
MRR: 0.17975
F1: 0.2433719354392884
Accuracy: 0.17975


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: False


In [ ]:
from collections import Counter
Counter(train_lab)

Counter({'NO_ANSWER': 14000})